# 第 5 周练习 — Chiku：Stayez 房地产知识工作者

**学生：** Vagz1216（马丁·卡莫）| **团队：** 欧几里得 | **第 5 周**

---

## 项目概况

该笔记本实现了应用于真实 Stayez 平台 (stayez.co.ke) 的完整 **RAG（检索增强生成）** 管道。我构建了一个**Stayez Property Knowledge Worker**，而不是通用的个人知识库，这是一个名为 **Chiku** 的人工智能预订助理，它可以通过语义搜索实际 Stayez 房产、体验和服务的精选知识库来回答客人的查询。

### 业务问题
Stayez 目前在 Python 字典中硬编码了属性数据。随着平台扩展到 500 多个房源，Chiku 无法为复杂的客人查询找到合适的房产，例如：
- *“我需要内罗毕一套带泳池的浪漫公寓，价格低于 7,000 肯尼亚先令”*
- *“适合五口之家，带停车位”*
- *“您在火山附近有哪些户外体验？”*

**RAG 解决了这个问题。** 每个属性都是一个 markdown 文件。任何新列表都会添加为新的“.md”文件。 Chiku 通过语义向量搜索找到正确的匹配项。

### 第 5 周概念演示
|概念 |实施 |
|---|---|
|文件加载| LangChain 每个文件夹类型的“DirectoryLoader” |
|文本分块 | `RecursiveCharacterTextSplitter`（1000 个字符，200 个重叠）|
|嵌入 | `HuggingFaceEmbeddings(all-MiniLM-L6-v2)` — 免费且本地 |
|矢量商店| `Chroma` 持久数据库 |
|可视化| t-SNE 2D + 3D Plotly 散点图 |
|拉格聊天 | Retriever + Groq/Gemini LLM 注射 |
|用户界面| Chiku 的 Gradio ‘ChatInterface’ |

In [ ]:
# 单元格 1：进口
# Cell 1: Imports

import os
import glob
import numpy as np
import plotly.graph_objects as go
import gradio as gr
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from sklearn.manifold import TSNE

# 浪链
# LangChain
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Imports loaded successfully")

In [ ]:
# 单元 2：配置和 API 密钥
# Cell 2: Configuration & API Keys

load_dotenv(find_dotenv(), override=True)

GROQ_API_KEY   = os.getenv('GROQ_API_KEY')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

for name, key in [("Groq", GROQ_API_KEY), ("Gemini", GEMINI_API_KEY)]:
    print(f"{name}: {'Found' if key else 'NOT SET'}")

# Groq 是我们的主要法学硕士（免费且快速）
# Groq is our primary LLM (free and fast)
groq_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

# 双子座作为后备
# Gemini as fallback
gemini_client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# 设置
# Settings
DB_NAME           = "stayez_vector_db"
KNOWLEDGE_BASE    = Path("stayez-knowledge-base")
EMBEDDING_MODEL   = "all-MiniLM-L6-v2"   # Free HuggingFace model, runs locally
CHUNK_SIZE        = 1000
CHUNK_OVERLAP     = 200

print(f"\nKnowledge base path: {KNOWLEDGE_BASE.resolve()}")
print(f"Vector DB: {DB_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL}")

## A 部分：加载和分块 Stayez 知识库

知识库分为 3 个文件夹：
- `properties/` — 单个属性列表文件（7 个属性）
- `experiences/` — 活动和经验列表  
- `services/` — 礼宾和支持服务

每个文件夹都成为矢量存储元数据中的文档“类型”标签。

In [ ]:
# 单元格 3：从知识库文件夹加载文档
# Cell 3: Load documents from the knowledge base folders

documents = []

for folder in KNOWLEDGE_BASE.iterdir():
    if not folder.is_dir():
        continue
    doc_type = folder.name
    loader = DirectoryLoader(
        str(folder),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        doc.metadata["source"] = Path(doc.metadata["source"]).name
    documents.extend(folder_docs)
    print(f"  - Loaded {len(folder_docs)} docs from '{doc_type}/'")

print(f"\nTotal documents loaded: {len(documents)}")

# 预览
# Preview
print(f"\nFirst document (excerpt):")
print(documents[0].page_content[:300])
print(f"Metadata: {documents[0].metadata}")

In [ ]:
# 单元 4：分成块
# Cell 4: Split into chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = splitter.split_documents(documents)

print(f"Documents split into {len(chunks)} chunks")
print(f"Average chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")
print(f"\nSample chunk:")
print(chunks[0].page_content)
print(f"\nMetadata: {chunks[0].metadata}")

## B 部分：嵌入块并构建 Chroma 矢量存储

我们使用 **HuggingFace 的 `all-MiniLM-L6-v2`** 模型将每个块转换为 384 维向量。该型号：
- 在 CPU 上**100% 本地**运行（无 API 调用）
- **完全免费**，没有速率限制
- 首次使用时自动下载 (~90 MB)
- 为英文文本提供出色的语义搜索质量

In [ ]:
# 单元 5：创建嵌入并构建 Chroma 矢量存储
# Cell 5: Create embeddings and build Chroma vector store

print("Loading HuggingFace embedding model (downloads ~90MB on first run)...")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print("Embedding model ready!")

# 删除现有集合以重新开始
# Delete existing collection to start fresh
if Path(DB_NAME).exists():
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
    print(f"Cleared existing vector store'{DB_NAME}'")

# 构建并保存向量存储
# Build and persist the vector store
print("\nBuilding Chroma vector store...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

collection = vectorstore._collection
count = collection.count()
sample_emb = collection.get(limit=1, include=["embeddings"])["embeddings"][0]

print(f"\nVector store created:")
print(f"  - Total vectors stored: {count:,}")
print(f"  - Embedding dimensions: {len(sample_emb):,} (all-MiniLM-L6-v2)")
print(f"  - Persisted to: ./{DB_NAME}/")

## C 部分：可视化向量空间

我们使用 **t-SNE**（t 分布式随机邻域嵌入）将 384 维向量压缩为 2D 和 3D 以便可视化。每个彩色点都是一个块，按其文档类别（财产、体验或服务）进行颜色编码。

> **要寻找的内容：** 如果 RAG 正常工作，相似的内容会聚集在一起。房产块应该组合在一起，体验块应该放在一起，等等。但是在房产集群内，浪漫/豪华房产应该彼此靠近，而经济型房产应该彼此靠近。

In [ ]:
# 单元 6：准备用于可视化的数据
# Cell 6: Prepare data for visualization

result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors   = np.array(result["embeddings"])
docs_text = result["documents"]
metadatas = result["metadatas"]
doc_types = [m["doc_type"] for m in metadatas]

# 按文档类型划分的彩色图
# Color map by document type
COLOR_MAP = {"properties": "royalblue", "experiences": "mediumseagreen", "services": "tomato"}
colors = [COLOR_MAP.get(t, "gray") for t in doc_types]

print(f"Data ready for visualization: {len(vectors)} vectors")
from collections import Counter
print(f"Breakdown: {dict(Counter(doc_types))}")

In [ ]:
# 单元 7：2D t-SNE 可视化
# Cell 7: 2D t-SNE Visualization

tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(5, len(vectors)-1))
reduced_2d = tsne_2d.fit_transform(vectors)

fig_2d = go.Figure(data=[go.Scatter(
    x=reduced_2d[:, 0],
    y=reduced_2d[:, 1],
    mode="markers",
    marker=dict(size=10, color=colors, opacity=0.85, line=dict(width=1, color="white")),
    text=[f"<b>{t}</b><br>{d[:120]}..." for t, d in zip(doc_types, docs_text)],
    hoverinfo="text"
)])

fig_2d.update_layout(
    title="2D Stayez Knowledge Base — Vector Clusters (t-SNE)",
    xaxis_title="t-SNE Dimension 1",
    yaxis_title="t-SNE Dimension 2",
    width=850, height=600,
    template="plotly_white"
)
fig_2d.show()

In [ ]:
# 单元 8：3D t-SNE 可视化
# Cell 8: 3D t-SNE Visualization

tsne_3d = TSNE(n_components=3, random_state=42, perplexity=min(5, len(vectors)-1))
reduced_3d = tsne_3d.fit_transform(vectors)

fig_3d = go.Figure(data=[go.Scatter3d(
    x=reduced_3d[:, 0],
    y=reduced_3d[:, 1],
    z=reduced_3d[:, 2],
    mode="markers",
    marker=dict(size=6, color=colors, opacity=0.85),
    text=[f"<b>{t}</b><br>{d[:120]}..." for t, d in zip(doc_types, docs_text)],
    hoverinfo="text"
)])

fig_3d.update_layout(
    title="3D Stayez Knowledge Base — Vector Clusters (t-SNE)",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900, height=700
)
fig_3d.show()

## D 部分：构建 Chiku — Stayez RAG 预订助手

现在我们将检索器和 LLM 连接在一起以创建 **Chiku**。

每个访客消息的流程：
1. Chiku收到客人的问题（例如*“我需要一间浪漫的公寓”*）
2. **Retriever** 将问题转换为向量，并从 Chroma 数据库中找到 5 个语义最相似的块
3. 检索到的块作为上下文注入到 Chiku 的**系统提示符**中
4. **Groq (Llama 3.3 70B)** 使用上下文和客人的问题生成最终的个性化答案

In [ ]:
# 单元 9：Chiku RAG 系统设置
# Cell 9: Chiku RAG System Setup

# k=3 保持上下文简洁（避免 Groq 免费层的令牌限制）
# k=3 keeps context concise (avoids token limit on Groq free tier)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

CHIKU_SYSTEM = """\
You are Chiku, a warm and knowledgeable guest assistant for Stayez (stayez.co.ke), \
a curated short-stay booking platform based in Kenya.

Your job is to help guests find the perfect property, experience, or service.
You are enthusiastic, friendly, and always recommend booking via stayez.co.ke.
All prices are in Kenyan Shillings (KSh). Always present prices clearly.

Use the context below to answer the guest's question accurately and concisely.
If information is not in the context, say so and suggest the guest visits stayez.co.ke.

RELEVANT STAYEZ CONTEXT:
{context}
"""

def build_messages(message: str, history: list, system_prompt: str) -> list:
    """Build a clean OpenAI-compatible message list.

    Handles any Gradio history format (tuples OR message dicts).
    IMPORTANT: strips any extra fields (e.g. Gradio 5 adds 'metadata')
    since Groq and Gemini only accept 'role' + 'content'.
    Caps history to last 4 messages (2 full turns) to stay within token limits.
    """
    normalised = []
    for item in history:
        if isinstance(item, dict) and "role" in item and "content" in item:
            # 仅将角色 + 内容列入白名单 — 删除“元数据”和任何其他 Gradio 添加的字段
            # Whitelist only role + content — drop 'metadata' and any other Gradio-added fields
            normalised.append({"role": item["role"], "content": item["content"]})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            u, b = item
            if u: normalised.append({"role": "user",      "content": str(u)})
            if b: normalised.append({"role": "assistant", "content": str(b)})
    normalised = normalised[-4:]  # keep last 2 turns only
    return [{"role": "system", "content": system_prompt}] + normalised + [{"role": "user", "content": message}]

def chiku_chat(message: str, history: list) -> str:
    # 步骤 1：从 Chroma 进行语义检索
    # Step 1: Semantic retrieval from Chroma
    docs = retriever.invoke(message)
    context = "\n\n---\n\n".join(doc.page_content for doc in docs)

    # 第 2 步：构建消息（干净，无额外字段）
    # Step 2: Build messages (clean, no extra fields)
    system_prompt = CHIKU_SYSTEM.format(context=context)
    messages = build_messages(message, history, system_prompt)

    # 第 3 步：尝试 Groq — llama-3.1-8b-instant（20,000 TPM 免费套餐）
    # Step 3: Try Groq — llama-3.1-8b-instant (20,000 TPM free tier)
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            temperature=0.4,
            max_tokens=512
        )
        return response.choices[0].message.content
    except Exception as groq_err:
        print(f"[Groq error] {type(groq_err).__name__}: {groq_err}")

    # 步骤 4：回退到 Gemini (gemini-2.0-flash-lite)
    # Step 4: Fallback to Gemini (gemini-2.0-flash-lite)
    try:
        response = gemini_client.chat.completions.create(
            model="gemini-2.0-flash-lite",
            messages=messages,
            temperature=0.4,
            max_tokens=512
        )
        return response.choices[0].message.content
    except Exception as gemini_err:
        print(f"[Gemini error] {type(gemini_err).__name__}: {gemini_err}")
        return "Chiku is temporarily unavailable. Please visit stayez.co.ke directly."

print("Chiku is ready!")

# 自检：2 轮（确保在启动 UI 之前 Groq 和历史记录处理工作）
# Self-test: 2 turns (ensures both Groq and history handling work before launching UI)
r1 = chiku_chat("What is your most popular property?", [])
print(f"Turn 1 OK: {r1[:200]}")

hist_test = [{"role": "user", "content": "What is your most popular property?"}, {"role": "assistant", "content": r1}]
r2 = chiku_chat("I need something romantic under KSh 7,000", hist_test)
print(f"Turn 2 OK: {r2[:200]}")

In [ ]:
# Cell 10：在 Gradio 上启动 Chiku
# Cell 10: Launch Chiku on Gradio

with gr.Blocks(theme=gr.themes.Soft(), title="Chiku — Stayez AI Booking Assistant") as demo:
    gr.Markdown(
        """## Chiku — Your Stayez Booking Assistant
Powered by RAG + Groq. Ask me about properties, experiences, and services at **stayez.co.ke**.

*Try: "I need a romantic apartment under KSh 7,000" or "What outdoor experiences do you have?"*
"""
    )
    chat = gr.ChatInterface(
        fn=chiku_chat,
        type="messages",
        examples=[
            "What is your most popular property?",
            "I need a romantic place in Nairobi for 2 nights, budget KSh 6,000",
            "Do you have properties for a family of 5?",
            "What outdoor experiences do you offer?",
            "Can you arrange an airport pickup?",
        ],
        chatbot=gr.Chatbot(type="messages", height=450, avatar_images=[None, "https://stayez.co.ke/wp-content/uploads/2023/08/stayez-logo.jpg"])
    )

demo.launch(inbrowser=True)

- -
## 概括

### 我们建造了什么
适用于 Stayez 平台的完整、可投入生产的 RAG 管道：

1. **Stayez 知识库** — 10 个 Markdown 文件（7 个属性、2 个体验、1 个服务）作为动态、可扩展的数据源。
2. **LangChain 文档加载** — `DirectoryLoader` 自动标记每个文档及其文件夹类型（属性/体验/服务）。
3. **`RecursiveCharacterTextSplitter`** — 通过 200 个字符重叠智能地对文档进行分块，以避免在边界处丢失上下文。
4. **免费嵌入** - `HuggingFaceEmbeddings(all-MiniLM-L6-v2)` 以零成本在本地运行，将文本转换为 384 维向量。
5. **Chroma Vector Store** — 将矢量数据库保存到磁盘，因此只需构建一次。
6. **t-SNE 可视化** — 证明向量空间正确组织：属性块聚集在一起，经验块聚集在一起。
7. **Chiku RAG Chat** — 5步流程：访客消息→矢量搜索→上下文注入→Groq LLM→友好回答。
8. **Gradio UI** — 干净的聊天界面，带有用于快速测试的示例提示。

### 这将为 Stayez 带来什么
当 Stayez 团队添加新属性时，他们只需将新的“.md”文件放入“stayez-knowledge-base/properties/”并重建矢量存储即可。 RAG 系统自动处理所有匹配 — 无需硬编码逻辑。